# Auto EDA: figures of Lesson 06.01

- This notebook generates the figures and tables of the slides
  `Lesson06.01_Auto_EDA.smd` from simulated data, and saves the figures as
  PNG files
- The running example is a daily equity panel, the input of the momentum
  strategy of Lessons 01.01 and 02.01, with planted defects
- The pedagogical arc:
  - Automated profiling: coverage of a panel by asset and date
  - Finance-specific checks: stale prices and unadjusted splits
  - Time-aware EDA: structural breaks and regime-aware summaries
  - Stylized facts as tests of the data
  - EDA as snooping: every look is a trial

In [1]:
%load_ext autoreload
%autoreload 2

import logging

In [2]:
import helpers.hintrospection as hintros
import helpers.hmatplotlib as hmatplo
import helpers.hnotebook as hnotebook

import L06_01_auto_eda_utils as utils

# Initialize notebook configuration and logging.
hnotebook.config_notebook()
_LOG = logging.getLogger(__name__)
utils.init_loggers(_LOG)

# Convert `display` into `print()` when running outside IPython.
try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore

INFO  > /Users/saggese/src/venv/client_venv.helpers/lib/python3.11/site-packages/ipykernel_launcher.py -f /private/var/folders/pd/0_nppqrn3cn7936mmhk37gcw0000gn/T/tmpi1im1xyv.json --HistoryManager.hist_file=:memory:


In [3]:
# Directory of the figures, and prefix of the markdown reference to paste in the
# slides.
out_dir_name = "figures"
markdown_path_prefix = "book_Modern_AI_for_Finance/lectures_source"

# Figure sizes: a single panel fits a column of a slide, two stacked panels
# need more height.
figsize = (6.0, 4.0)
figsize_tall = (6.0, 5.0)

# Seed of all the simulations.
seed = 6

# Part 1: Automated profiling

## Cell 1.1: Panel coverage by asset and date

**Goal**
- Simulate the daily equity panel of the running example
- See listings, delistings, and a vendor outage that a column-by-column
  profiler hides
- See that a survivor-only database never shows a stock that exits

**Description**
- Inputs
  - `n_stocks`: number of stocks
  - `n_days`: number of business days
  - `ipo_fraction`: fraction of the stocks that list after the start
  - `delisting_price`: price, relative to the listing price, that triggers
    a delisting
  - `outage_fraction`, `outage_start`, `outage_days`: share of stocks, first
    day, and length of the vendor outage
  - `seed`: seed of the random number generator

- Panels
  - `Coverage`: one row per stock sorted by listing date, one column per
    month, darker for a larger fraction of days with a price
  - `Stocks with a price`: count per day in the full database in blue and in
    a survivor-only database in red

In [4]:
# Simulate the panel of close prices.
prices_df = utils.simulate_panel(
    n_stocks=200,
    n_days=2520,
    ipo_fraction=0.3,
    delisting_price=0.3,
    outage_fraction=0.25,
    outage_start=1300,
    outage_days=20,
    seed=seed,
)
print("prices_df.shape=", prices_df.shape)
display(prices_df.iloc[-5:, :5])

prices_df.shape= (2520, 200)


,S000,S001,S002,S003,S004
2024-08-22,18.25,NaN,NaN,NaN,NaN
2024-08-23,18.02,NaN,NaN,NaN,NaN
2024-08-26,18.06,NaN,NaN,NaN,NaN
2024-08-27,18.21,NaN,NaN,NaN,NaN
2024-08-28,18.64,NaN,NaN,NaN,NaN


In [5]:
# Plot the coverage of the panel and the number of stocks with a price.
fig = utils.cell1_1_plot_panel_coverage(prices_df, figsize=figsize_tall)
hmatplo.save_fig(
    fig,
    f"{out_dir_name}/L06.01.Panel_Coverage.png",
    print_markdown=True,
    path_prefix=markdown_path_prefix,
)

Saved figure to 'figures/L06.01.Panel_Coverage.png'
![](book_Modern_AI_for_Finance/lectures_source/figures/L06.01.Panel_Coverage.png)


**Guided usage**
- Change `delisting_price` from 0.3 to 0.5
  - Observe more stocks delist, and the gap between the blue and the red
    curves widen in the early years
- Change `outage_days` from 20 to 60
  - Observe a longer dip in the count and a lighter band in the heatmap

**Implementation** `cell1_1_plot_panel_coverage()`
- Computes the fraction of business days with a price for each stock and
  month
- Sorts the stocks by first and last date with a price
- Keeps in the survivor-only database the stocks with a price on the last
  day

In [6]:
hintros.print_obj_info(utils.cell1_1_plot_panel_coverage)

- `cell1_1_plot_panel_coverage(prices_df: pandas.core.frame.DataFrame, *, figsize: Optional[Tuple[float, float]] = None) -> matplotlib.figure.Figure`  
  Plot the coverage of a panel by stock and month, and the listed stocks.

https://github.com/gpsaggese/gpsaggese.github.io/blob/gp/book_Modern_AI_for_Finance/tutorials/L06_auto_eda/L06_01_auto_eda_utils.py#L219


# Part 2: Finance-specific checks

## Cell 2.1: Stale prices

**Goal**
- Make the prices of some illiquid stocks stale, as when a stock does not
  trade and the database repeats the last price
- Detect stale prices with the fraction of zero returns and the longest flat
  run
- See that stale prices make an equal-weighted index look predictable

**Description**
- Inputs
  - `stale_stocks`: stocks whose prices become stale
  - `stale_prob`: probability that a stale stock does not trade on a day
  - `zero_fraction_threshold`: fraction of zero returns above which a stock
    is flagged

- Panels
  - `Price`: one stale stock, with the true price in gray, the recorded price
    in blue, and the zero returns in red
  - `Detector`: fraction of zero returns against the longest flat run for
    every stock, with the stale stocks in red

In [7]:
# Make the prices of the first 40 stocks stale.
stale_stocks = list(prices_df.columns[:40])
stale_df = utils.add_stale_prices(
    prices_df, stocks=stale_stocks, stale_prob=0.4, seed=seed
)

In [8]:
# Plot one stale stock and the detector.
fig = utils.cell2_1_plot_stale_prices(
    prices_df,
    stale_df,
    stale_stocks=stale_stocks,
    stock=stale_stocks[0],
    n_days_shown=60,
    zero_fraction_threshold=0.1,
    figsize=figsize_tall,
)
hmatplo.save_fig(
    fig,
    f"{out_dir_name}/L06.01.Stale_Prices.png",
    print_markdown=True,
    path_prefix=markdown_path_prefix,
)

Saved figure to 'figures/L06.01.Stale_Prices.png'
![](book_Modern_AI_for_Finance/lectures_source/figures/L06.01.Stale_Prices.png)


In [9]:
# Compare the lag-1 autocorrelation of an equal-weighted index.
index_acf = {
    "clean": utils.compute_index_autocorrelation(prices_df),
    "stale": utils.compute_index_autocorrelation(stale_df),
}
print("index_acf=", index_acf)

index_acf= {'clean': 0.016586124457674413, 'stale': 0.07073064320806206}


**Guided usage**
- Change `stale_prob` from 0.4 to 0.1
  - Observe the stale stocks move toward the cloud of stocks that trade
    daily, and the index autocorrelation shrink
- Change `stale_stocks` to the first 100 stocks
  - Observe the lag-1 autocorrelation of the index grow: the stale stocks
    catch up with the market move of the previous day

**Implementation** `cell2_1_plot_stale_prices()`
- Plots the recorded price as a step function, so that a flat run is visible
- Computes, for each stock, the fraction of returns equal to 0 and the
  longest run of zero returns with `compute_stale_stats()`

In [10]:
hintros.print_obj_info(utils.cell2_1_plot_stale_prices)

- `cell2_1_plot_stale_prices(clean_df: pandas.core.frame.DataFrame, stale_df: pandas.core.frame.DataFrame, *, stale_stocks: List[str], stock: str, n_days_shown: int, zero_fraction_threshold: float, figsize: Optional[Tuple[float, float]] = None) -> matplotlib.figure.Figure`  
  Plot one stock with stale prices and a stale-price detector.

https://github.com/gpsaggese/gpsaggese.github.io/blob/gp/book_Modern_AI_for_Finance/tutorials/L06_auto_eda/L06_01_auto_eda_utils.py#L383


## Cell 2.2: Unadjusted split

**Goal**
- Plant a 2-for-1 split that the database does not adjust for
- Detect it with a robust z-score and the rule that an unadjusted split with
  ratio `k` produces a return of `1 / k - 1`

**Description**
- Inputs
  - `split_stock`, `split_date`, `ratio`: stock, date, and ratio of the split
  - `z_threshold`: absolute robust z-score above which a return is flagged
  - `ratios`: split ratios to test
  - `tolerance`: largest distance between a return and `1 / k - 1`

- Panels
  - `Price`: adjusted price in blue and raw price in red
  - `Returns`: daily returns of the raw price, with the robust band dashed
    and the split day in red

In [11]:
# Plant an unadjusted 2-for-1 split in a stock listed on the whole period.
split_stock = "S150"
raw_df = utils.add_unadjusted_split(
    prices_df, stock=split_stock, split_date="2019-06-03", ratio=2.0
)

In [12]:
# Plot the split and the robust band.
fig = utils.cell2_2_plot_split_detection(
    prices_df,
    raw_df,
    stock=split_stock,
    ratio=2.0,
    z_threshold=8.0,
    figsize=figsize_tall,
)
hmatplo.save_fig(
    fig,
    f"{out_dir_name}/L06.01.Split_Detection.png",
    print_markdown=True,
    path_prefix=markdown_path_prefix,
)

Saved figure to 'figures/L06.01.Split_Detection.png'
![](book_Modern_AI_for_Finance/lectures_source/figures/L06.01.Split_Detection.png)


In [13]:
# Scan the whole panel for extreme returns and split ratios.
candidates_df = utils.detect_split_candidates(
    raw_df, z_threshold=8.0, ratios=[2.0, 3.0, 1.5, 4.0], tolerance=0.01
)
print("n_flagged=", len(candidates_df))
print("n_split_like=", int(candidates_df["split_ratio"].notna().sum()))
display(candidates_df.sort_values("robust_z").head(10))

n_flagged= 176
n_split_like= 1


,date,stock,return,robust_z,split_ratio
69,2019-06-03,S150,-0.501363,-38.077479,2.0
77,2020-05-06,S127,-0.260978,-18.560381,NaN
93,2021-02-08,S137,-0.356699,-18.038801,NaN
20,2016-03-04,S145,-0.224450,-15.862264,NaN
15,2015-10-15,S061,-0.289830,-15.363526,NaN
84,2020-08-28,S100,-0.277529,-15.130274,NaN
9,2015-04-20,S062,-0.256750,-14.951891,NaN
101,2021-07-30,S141,-0.192619,-14.023159,NaN
61,2018-10-23,S132,-0.246842,-12.583107,NaN
82,2020-07-16,S114,-0.217803,-12.557829,NaN


**Guided usage**
- Change `z_threshold` from 8 to 5
  - Observe many more flagged returns: heavy tails make extreme returns
    common, and only the split-ratio rule separates the split from a crash
- Change `ratio` from 2 to 1.5
  - Observe a -33.3% return, which matches a 3-for-2 split

**Implementation** `cell2_2_plot_split_detection()`
- Computes the band as the median return plus or minus `z_threshold` times
  the scaled MAD
- Marks the most negative return as the split candidate

In [14]:
hintros.print_obj_info(utils.cell2_2_plot_split_detection)

- `cell2_2_plot_split_detection(adjusted_df: pandas.core.frame.DataFrame, raw_df: pandas.core.frame.DataFrame, *, stock: str, ratio: float, z_threshold: float, figsize: Optional[Tuple[float, float]] = None) -> matplotlib.figure.Figure`  
  Plot a stock with an unadjusted split and the robust return band.

https://github.com/gpsaggese/gpsaggese.github.io/blob/gp/book_Modern_AI_for_Finance/tutorials/L06_auto_eda/L06_01_auto_eda_utils.py#L554


# Part 3: Time-aware EDA

## Cell 3.1: Structural breaks with CUSUM

**Goal**
- Simulate a strategy with a volatility regime and a later alpha decay
- See that a CUSUM test detects the break in variance but not the break in
  mean

**Description**
- Inputs
  - `segments`: list of `(n_days, annual_sharpe, annual_vol)` for each regime
  - `seed`: seed of the random number generator

- Panels
  - `Cumulative return`: cumulative return of the strategy, with the true
    breaks dotted
  - `CUSUM`: absolute CUSUM statistic of a break in mean in blue and of a
    break in variance in red, with the 5% critical value dashed

In [15]:
# Simulate 3 calm years, 1.5 crisis years, 1.5 calm years, and 4 years in
# which the edge is gone.
segments = [
    (756, 1.0, 0.10),
    (378, 1.0, 0.20),
    (378, 1.0, 0.10),
    (1008, 0.0, 0.10),
]
strategy_returns = utils.simulate_strategy_returns(segments=segments, seed=seed)
break_days = [756, 1134, 1512]

In [16]:
# Plot the cumulative return and the CUSUM statistics.
fig = utils.cell3_1_plot_structural_breaks(
    strategy_returns, break_days=break_days, figsize=figsize_tall
)
hmatplo.save_fig(
    fig,
    f"{out_dir_name}/L06.01.Structural_Breaks.png",
    print_markdown=True,
    path_prefix=markdown_path_prefix,
)

Saved figure to 'figures/L06.01.Structural_Breaks.png'
![](book_Modern_AI_for_Finance/lectures_source/figures/L06.01.Structural_Breaks.png)


**Guided usage**
- Change the Sharpe ratio of the first three regimes from 1.0 to 2.0
  - Observe the mean CUSUM cross the 5% line: only a large drop in the mean is
    visible in 10 years of daily data
- Change the volatility of the crisis from 0.20 to 0.12
  - Observe the variance CUSUM still cross the line: breaks in volatility are
    much easier to detect than breaks in mean

**Implementation** `cell3_1_plot_structural_breaks()`
- Computes the OLS-CUSUM of the demeaned returns and the cumulative sum of
  squares with `compute_cusum_stats()`
- Compares the maximum of each absolute statistic with the 5% critical value
  1.358 of a Brownian bridge

In [17]:
hintros.print_obj_info(utils.cell3_1_plot_structural_breaks)

- `cell3_1_plot_structural_breaks(returns: pandas.core.series.Series, *, break_days: List[int], figsize: Optional[Tuple[float, float]] = None) -> matplotlib.figure.Figure`  
  Plot the cumulative return of a strategy and its CUSUM statistics.

https://github.com/gpsaggese/gpsaggese.github.io/blob/gp/book_Modern_AI_for_Finance/tutorials/L06_auto_eda/L06_01_auto_eda_utils.py#L678


## Cell 3.2: Regime-aware summaries

**Goal**
- Compare the summary of the full sample with the summaries by regime

In [18]:
# Summarize the strategy over the full sample and by regime.
summary_df = utils.cell3_2_summarize_by_period(
    strategy_returns,
    boundaries=break_days,
    labels=["calm", "crisis", "calm again", "edge gone"],
)
display(summary_df.round(2))

,years,annual_return,annual_vol,sharpe
period,,,,
full sample,10.0,0.10,0.12,0.84
calm,3.0,0.07,0.10,0.76
crisis,1.5,0.23,0.21,1.10
calm again,1.5,0.23,0.10,2.26
edge gone,4.0,0.03,0.10,0.28


# Part 4: Stylized facts

## Cell 4.1: Stylized facts test suite

**Goal**
- Compute the stylized-fact statistics of a clean index and of four
  corrupted versions
- See that each data defect breaks a different stylized fact

In [19]:
# Build the clean and corrupted price series.
corrupted_prices = utils.build_corrupted_prices(
    n_days=2520, missing_prob=0.3, split_ratio=2.0, seed=seed
)

In [20]:
# Compute the test suite on each version.
facts_df = utils.cell4_1_compute_stylized_facts_table(
    corrupted_prices, max_lag=10
)
display(facts_df.round(3))

,clean,stale,interpolated,shuffled,unadjusted split
excess_kurtosis_daily,8.701,23.439,12.771,8.706,653.410
excess_kurtosis_monthly,2.028,1.688,2.174,-0.065,35.244
acf1_returns,0.024,-0.024,0.172,-0.021,0.012
acf_abs_returns,0.185,0.079,0.176,-0.000,0.062
zero_return_fraction,0.000,0.310,0.000,0.000,0.000


# Part 5: EDA as snooping

## Cell 5.1: Every look is a trial

**Goal**
- Test more and more noise features against noise returns, and keep the
  feature with the best in-sample IC
- See the best IC grow with the number of looks and vanish on the holdout

**Description**
- Inputs
  - `n_in`, `n_out`: number of in-sample and holdout days
  - `looks`: numbers of looks to evaluate
  - `n_reps`: number of repetitions of the simulation

- Panels
  - `IC`: mean IC of the best feature in-sample in blue and on the holdout in
    red, with the 10%-90% band shaded and the single-test 5% threshold dashed

In [21]:
# Simulate the best of up to 1,000 noise features, 5 years in-sample and 5
# years of holdout.
n_in = 1260
looks = [1, 3, 10, 30, 100, 300, 1000]
snooping_df = utils.simulate_snooping(
    n_in=n_in, n_out=1260, looks=looks, n_reps=50, seed=seed
)
display(snooping_df.groupby("looks")[["ic_in", "ic_out"]].mean().round(3))

,ic_in,ic_out
looks,,
1,0.022,-0.002
3,0.038,0.003
10,0.052,0.006
30,0.064,0.005
100,0.079,0.005
300,0.089,-0.002
1000,0.097,0.008


In [22]:
# Plot the IC of the selected feature against the number of looks.
fig = utils.cell5_1_plot_snooping(snooping_df, n_in=n_in, figsize=figsize)
hmatplo.save_fig(
    fig,
    f"{out_dir_name}/L06.01.Snooping_Looks.png",
    print_markdown=True,
    path_prefix=markdown_path_prefix,
)

Saved figure to 'figures/L06.01.Snooping_Looks.png'
![](book_Modern_AI_for_Finance/lectures_source/figures/L06.01.Snooping_Looks.png)


**Guided usage**
- Change `n_in` from 1260 to 5040
  - Observe the best in-sample IC shrink by half: a 4x longer history halves
    the noise, but does not remove the selection effect
- Change `looks` to stop at 10
  - Observe the best IC stay close to the single-test threshold

**Implementation** `cell5_1_plot_snooping()`
- Simulates independent Gaussian targets and features with
  `simulate_snooping()`, so every true IC is 0
- Keeps, for each number of looks, the feature with the largest absolute
  in-sample IC, and measures it on the holdout with the in-sample sign

In [23]:
hintros.print_obj_info(utils.cell5_1_plot_snooping)

- `cell5_1_plot_snooping(snooping_df: pandas.core.frame.DataFrame, *, n_in: int, figsize: Optional[Tuple[float, float]] = None) -> matplotlib.figure.Figure`  
  Plot the in-sample and holdout IC of the best feature against the looks.

https://github.com/gpsaggese/gpsaggese.github.io/blob/gp/book_Modern_AI_for_Finance/tutorials/L06_auto_eda/L06_01_auto_eda_utils.py#L946
